In [5]:
import pandas as pd

"""
The dataset is a structured compilation based on:
1. Human Rights Activists News Agency (HRANA) daily updates and comprehensive reports (Primary source)
   Link: https://www.en-hrana.org/a-comprehensive-report-of-the-first-82-days-of-nationwide-protests-in-iran/
2. Daily Statistics of the 2022 Iran Protests (GitHub repository compiling HRANA data)
   Link: https://github.com/justin-2028/Daily-Statistics-of-the-2022-Iran-Protests
3. UK Parliament Commons Library Briefing (CBP-9679) referencing HRANA statistics
   Link: https://commonslibrary.parliament.uk/research-briefings/cbp-9679/

The Sankey categories (arrests, fatalities, identified detainees, prison sentences,
and executions) are derived from the cumulative figures reported in these sources.
"""
print("=== Generating Chart 9 Data ===")

# The numbers below are based on the documented sources outlined above.
sankey_data = [
    # 1. First flow: From the streets to where? (Based on HRANA cumulative estimates)
    {'source': 'Protesters', 'target': 'Killed', 'value': 537},
    {'source': 'Protesters', 'target': 'Arrested', 'value': 19200},

    # 2. Second flow: The fate of the arrested (Derived from HRANA statistics)
    # Out of ~19,200 estimated arrests, only a fraction were officially registered/identified.
    # The rest represent undocumented detentions ("The Void").
    {'source': 'Arrested', 'target': 'Identified / Tracked', 'value': 5209},
    {'source': 'Arrested', 'target': 'Status Unknown (The Void)', 'value': 13991},

    # 3. Third flow: The fate of the identified/tracked individuals
    # Breaking down the identified detainees into prison sentences vs. those released/acquitted.
    {'source': 'Identified / Tracked', 'target': 'Released on Bail', 'value': 4433}, # 5209 - 776
    {'source': 'Identified / Tracked', 'target': 'Sentenced to Prison', 'value': 776},

    # 4. Fourth flow: The fate of the prisoners (Based on HRANA & Amnesty International)
    # Tracking executions vs. those remaining incarcerated.
    {'source': 'Sentenced to Prison', 'target': 'Executed', 'value': 8},
    {'source': 'Sentenced to Prison', 'target': 'Still in Prison', 'value': 768} # 776 - 8
]

# Convert the structured data to a pandas DataFrame
df_sankey = pd.DataFrame(sankey_data)

# Save the file
output_file_9 = 'chart9_sankey.csv'
df_sankey.to_csv(output_file_9, index=False)

print("\n Sankey data (Chart 9) generated successfully with documented methodology!")
print(df_sankey)

=== Generating Chart 9 Data ===

 Sankey data (Chart 9) generated successfully with documented methodology!
                 source                     target  value
0            Protesters                     Killed    537
1            Protesters                   Arrested  19200
2              Arrested       Identified / Tracked   5209
3              Arrested  Status Unknown (The Void)  13991
4  Identified / Tracked           Released on Bail   4433
5  Identified / Tracked        Sentenced to Prison    776
6   Sentenced to Prison                   Executed      8
7   Sentenced to Prison            Still in Prison    768


In [3]:
import pandas as pd
import calendar

print("=== Chart 10: Heatmap  ===")

"""
Primary Base Source:
- ACLED (Armed Conflict Location & Event Data Project)
  Filtered strictly for 'Protests' and 'Riots' to isolate civilian uprising data.
"""

# 1. Load the raw ACLED data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Filter Iran's data (2017 to 2025) AND filter for Protest/Repression events
df_me['DATE'] = pd.to_datetime(df_me['WEEK'])
df_iran = df_me[(df_me['COUNTRY'] == 'Iran') &
                (df_me['DATE'].dt.year >= 2017) &
                (df_me['DATE'].dt.year <= 2025) &
                (df_me['EVENT_TYPE'].isin(['Protests', 'Riots']))].copy()

df_iran['YEAR'] = df_iran['DATE'].dt.year
df_iran['MONTH'] = df_iran['DATE'].dt.month

# 3. Aggregate fatalities in each month and year (Only for the filtered events)
heatmap_data = df_iran.groupby(['YEAR', 'MONTH'])['FATALITIES'].sum().reset_index()

# 4. Pivot: Convert data into a matrix (Rows=Year, Columns=Month)
df_heatmap = heatmap_data.pivot(index='YEAR', columns='MONTH', values='FATALITIES').fillna(0)

# 5. Convert month numbers to short names for D3
df_heatmap.columns = [calendar.month_abbr[i] for i in df_heatmap.columns]
df_heatmap.reset_index(inplace=True)

# 6. Save the file
output_file_10 = 'chart10_heatmap.csv'
df_heatmap.to_csv(output_file_10, index=False)

print("\n Heatmap matrix is ready! (Filtered for State Repression)")
print(df_heatmap[['YEAR', 'Sep', 'Oct', 'Nov', 'Dec']])

=== Chart 10: Heatmap  ===

✅ Heatmap matrix is ready! (Filtered for State Repression)
   YEAR  Sep  Oct  Nov  Dec
0  2017    0    0    0   21
1  2018    0    0    0    0
2  2019    0    0  387    0
3  2020    0    1    0    0
4  2021    0    0    0    0
5  2022  231  104  134    6
6  2023    1    0    3    0
7  2024    0    1    5    0
8  2025    1    0    0   29


In [6]:
import pandas as pd

print("=== Chart 10 (Alternative): The Void Gap (ACLED vs Reality) ===")

"""
Data Sources – The Gap (November 2019 & Fall 2022):
1. Reuters (Dec 23, 2019): Estimated approximately 1,500 deaths during the November 2019
   protests, based on Iranian government sources.
   Link: https://www.reuters.com/article/us-iran-protests-specialreport-idUSKBN1YR0QR

2. NetBlocks (Nov 2019): Documented nationwide internet shutdowns beginning November 15, 2019.
   Link: https://netblocks.org/reports/internet-restored-in-iran-after-protest-shutdown-dAmqddA9

3. NetBlocks (Fall 2022): Documented widespread connectivity disruptions during the 2022
   Woman, Life, Freedom protests.
   Link: https://netblocks.org/reports/internet-disrupted-in-iran-amid-protests-over-death-of-mahsa-amini-X8qVEwAD

4. HRANA (2022): Estimated 537 deaths during the Woman, Life, Freedom protests.
"""
# 1. Load the ACLED data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)
df_me['DATE'] = pd.to_datetime(df_me['WEEK'])
df_me['YEAR_MONTH'] = df_me['DATE'].dt.to_period('M')

# 2. Extract ACLED statistics ONLY for Protest/Repression events (Matches Heatmap exactly)
df_iran = df_me[(df_me['COUNTRY'] == 'Iran') &
                (df_me['EVENT_TYPE'].isin(['Protests', 'Riots']))]

acled_monthly = df_iran.groupby('YEAR_MONTH')['FATALITIES'].sum().reset_index()
acled_monthly['YEAR_MONTH'] = acled_monthly['YEAR_MONTH'].astype(str)

# Reported fatalities by ACLED in Nov 2019 (Aban 98)
acled_nov_2019_data = acled_monthly[acled_monthly['YEAR_MONTH'] == '2019-11']['FATALITIES'].values
acled_nov_2019 = int(acled_nov_2019_data[0]) if len(acled_nov_2019_data) > 0 else 0

# Reported fatalities by ACLED in Fall 2022 (Sep-Dec 2022)
months_2022 = ['2022-09', '2022-10', '2022-11', '2022-12']
acled_fall_2022 = acled_monthly[acled_monthly['YEAR_MONTH'].isin(months_2022)]['FATALITIES'].sum()

# 3. Build the comparison dataset (The Void) with global documented sources
void_data = [
    {
        'Event': 'Aban Protests (Nov 2019)',
        'Internet_Blackout': 'Yes (NetBlocks)',
        'ACLED_Reported_Deaths': acled_nov_2019,
        'Real_Estimated_Deaths': 1500, # Source: Reuters
        'The_Void_Gap': 1500 - acled_nov_2019,
        'Source': 'Reuters / Amnesty'
    },
    {
        'Event': 'Mahsa Protests (Fall 2022)',
        'Internet_Blackout': 'Yes (NetBlocks)',
        'ACLED_Reported_Deaths': acled_fall_2022,
        'Real_Estimated_Deaths': 537, # Source: HRANA
        'The_Void_Gap': 537 - acled_fall_2022,
        'Source': 'HRANA'
    }
]

df_void_gap = pd.DataFrame(void_data)

# 4. Save the file
output_file_10_alt = 'chart10_void_gap.csv'
df_void_gap.to_csv(output_file_10_alt, index=False)

print("\n The Void Gap data is ready!")
print(df_void_gap[['Event', 'ACLED_Reported_Deaths', 'Real_Estimated_Deaths', 'The_Void_Gap']])

=== Chart 10 (Alternative): The Void Gap (ACLED vs Reality) ===

 The Void Gap data is ready!
                        Event  ACLED_Reported_Deaths  Real_Estimated_Deaths  \
0    Aban Protests (Nov 2019)                    387                   1500   
1  Mahsa Protests (Fall 2022)                    475                    537   

   The_Void_Gap  
0          1113  
1            62  
